# Computational Submodule 9: Privacy & Forensics in Blockchain

## Overview

This notebook provides hands-on experience with blockchain privacy analysis and forensic techniques. You will implement address clustering algorithms, transaction graph analysis, taint tracking, and privacy-enhancing technologies. All examples use **synthetic data** -- no external APIs or network access required.

**Prerequisites:**
- Completed [Notebook 01: Cryptographic Primitives](./01-cryptographic-primitives.ipynb) (hash functions, ECDSA, key generation)
- Completed [Notebook 02: Bitcoin Blockchain Analysis](./02-bitcoin-blockchain-analysis.ipynb) (transaction structure, UTXOs)
- Basic Python programming and familiarity with graph data structures

**Learning Objectives:**

By the end of this notebook, you will be able to:
1. Explain Bitcoin's pseudonymity model and demonstrate how address reuse degrades privacy
2. Build and analyze transaction graphs using NetworkX, identifying hubs and clusters
3. Implement the Common-Input-Ownership Heuristic (CIOH) and Union-Find-based address clustering
4. Perform taint analysis using both the poison/haircut and FIFO methods
5. Simulate CoinJoin transactions and evaluate their effect on privacy heuristics
6. Construct hash-based commitment schemes and analyze anonymity sets

**Estimated Time:** 4--6 hours

**Reference Reading:** [Section 6: Privacy Technologies](../sections/06-privacy-technologies.md)

---

## Setup and Imports

In [ ]:
# Standard library
import hashlib
import secrets
import random
import itertools
from collections import defaultdict, Counter
from typing import List, Dict, Tuple, Set

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

# Reproducibility
random.seed(42)
np.random.seed(42)

print("All imports successful.")

---
# Part 1: Bitcoin Privacy Model

Bitcoin is **pseudonymous**, not anonymous. Every transaction is recorded on the public blockchain, and the only protection is the disconnect between an address (a hash) and a real-world identity. In this section we generate synthetic Bitcoin-style addresses and explore how common usage patterns -- especially **address reuse** -- erode privacy.

### 1.1 Generating Simulated Bitcoin-Style Addresses

Real Bitcoin addresses are derived from public keys via SHA-256 + RIPEMD-160 + Base58Check. Here we simulate the process using SHA-256 to create 15 distinct addresses, each associated with a synthetic "user".

In [ ]:
def generate_address(seed_bytes: bytes) -> str:
    """Generate a simulated Bitcoin-style address from seed bytes."""
    h = hashlib.sha256(seed_bytes).hexdigest()
    # Simulate RIPEMD-160 step with another hash round, take 40 hex chars
    ripe = hashlib.new('ripemd160', bytes.fromhex(h)).hexdigest()
    return '1' + ripe[:33]  # Prefix '1' like legacy Bitcoin addresses

# Generate 15 addresses, each owned by one of 5 users (3 addresses per user)
users = ['Alice', 'Bob', 'Carol', 'Dave', 'Eve']
addresses = []
address_owner = {}  # ground truth mapping

for i, user in enumerate(users):
    for j in range(3):
        seed = f"{user}-keypair-{j}".encode()
        addr = generate_address(seed)
        addresses.append(addr)
        address_owner[addr] = user

print(f"Generated {len(addresses)} addresses for {len(users)} users:\n")
for addr in addresses:
    print(f"  {addr}  ->  {address_owner[addr]}")

### 1.2 Address Reuse and Privacy Degradation

When a user receives payments to the **same** address multiple times, an observer can trivially link those payments. Below we simulate a scenario where Alice reuses her first address for three incoming payments, while Bob uses a fresh address each time.

In [ ]:
alice_addrs = [a for a in addresses if address_owner[a] == 'Alice']
bob_addrs = [a for a in addresses if address_owner[a] == 'Bob']

# Alice reuses her first address for all incoming payments
alice_reuse_payments = [
    {'from': 'External-1', 'to': alice_addrs[0], 'amount': 0.5},
    {'from': 'External-2', 'to': alice_addrs[0], 'amount': 1.2},
    {'from': 'External-3', 'to': alice_addrs[0], 'amount': 0.3},
]

# Bob uses a fresh address for each payment
bob_fresh_payments = [
    {'from': 'External-4', 'to': bob_addrs[0], 'amount': 0.7},
    {'from': 'External-5', 'to': bob_addrs[1], 'amount': 0.4},
    {'from': 'External-6', 'to': bob_addrs[2], 'amount': 0.9},
]

# Analysis: count unique addresses seen per user from an observer's viewpoint
alice_observed = set(p['to'] for p in alice_reuse_payments)
bob_observed = set(p['to'] for p in bob_fresh_payments)

print("=== Observer's View ===")
print(f"\nAlice reuses addresses:")
print(f"  Unique receiving addresses observed: {len(alice_observed)}")
print(f"  Total payments linkable to one entity: {len(alice_reuse_payments)}")
print(f"  -> An observer KNOWS all 3 payments go to the same person.")

print(f"\nBob uses fresh addresses:")
print(f"  Unique receiving addresses observed: {len(bob_observed)}")
print(f"  Payments linkable without further analysis: 1 per address")
print(f"  -> An observer sees 3 apparently DIFFERENT recipients.")

### 1.3 Common-Input-Ownership Heuristic (CIOH)

The CIOH states: **if two or more addresses appear as inputs in the same transaction, they are likely controlled by the same entity.** This is because spending from an address requires the corresponding private key, and a single transaction typically means a single signer (or coordinated signers).

Below we build synthetic transactions where multiple inputs come from the same user, then apply CIOH to link addresses.

In [ ]:
# Synthetic transactions with multiple inputs
cioh_transactions = [
    # Alice spends from two of her addresses in one transaction
    {
        'txid': 'tx_cioh_01',
        'inputs': [alice_addrs[0], alice_addrs[1]],
        'outputs': [bob_addrs[0]],
        'amounts_in': [0.5, 0.3],
        'amounts_out': [0.75],  # 0.05 fee
    },
    # Carol spends from two of her addresses
    {
        'txid': 'tx_cioh_02',
        'inputs': [
            [a for a in addresses if address_owner[a] == 'Carol'][0],
            [a for a in addresses if address_owner[a] == 'Carol'][1],
        ],
        'outputs': [
            [a for a in addresses if address_owner[a] == 'Dave'][0]
        ],
        'amounts_in': [1.0, 0.5],
        'amounts_out': [1.45],
    },
]

def apply_cioh(transactions: list) -> list:
    """Apply CIOH: return list of sets of linked addresses."""
    linked_groups = []
    for tx in transactions:
        if len(tx['inputs']) > 1:
            linked_groups.append(set(tx['inputs']))
    return linked_groups

cioh_results = apply_cioh(cioh_transactions)

print("CIOH Analysis Results:")
print("="*60)
for i, group in enumerate(cioh_results):
    owners = set(address_owner[a] for a in group)
    print(f"\nLinked Group {i+1}:")
    for addr in group:
        print(f"  {addr[:20]}...  (true owner: {address_owner[addr]})")
    correct = len(owners) == 1
    print(f"  CIOH correct? {correct} (all owned by {owners})")

---
# Part 2: Transaction Graph Analysis

Blockchain transactions form a **directed graph** where addresses are nodes and value flows are edges. Analyzing this graph reveals spending patterns, hub addresses (e.g., exchanges), and relationships between entities.

### 2.1 Building Synthetic Transaction Data

We create 30 transactions among our 15 addresses with realistic amounts and structure.

In [ ]:
random.seed(42)

synthetic_txs = []
for i in range(30):
    # Pick random sender and receiver (different people)
    sender_user = random.choice(users)
    receiver_user = random.choice([u for u in users if u != sender_user])
    
    sender_addrs = [a for a in addresses if address_owner[a] == sender_user]
    receiver_addrs = [a for a in addresses if address_owner[a] == receiver_user]
    
    # Number of inputs: 1 or 2
    n_inputs = random.choice([1, 1, 1, 2])  # bias toward single input
    inputs = random.sample(sender_addrs, min(n_inputs, len(sender_addrs)))
    
    # Outputs: payment + optional change
    payment_addr = random.choice(receiver_addrs)
    amount = round(random.uniform(0.01, 2.0), 4)
    
    outputs = [payment_addr]
    out_amounts = [amount]
    
    # 60% chance of change output back to sender
    if random.random() < 0.6:
        change_addr = random.choice(sender_addrs)
        change_amount = round(random.uniform(0.001, 0.5), 4)
        outputs.append(change_addr)
        out_amounts.append(change_amount)
    
    synthetic_txs.append({
        'txid': f'tx_{i:03d}',
        'inputs': inputs,
        'outputs': outputs,
        'amounts_out': out_amounts,
    })

print(f"Created {len(synthetic_txs)} synthetic transactions.")
print(f"\nSample transaction:")
tx_sample = synthetic_txs[0]
print(f"  TXID: {tx_sample['txid']}")
print(f"  Inputs:  {[a[:16]+'...' for a in tx_sample['inputs']]}")
print(f"  Outputs: {[a[:16]+'...' for a in tx_sample['outputs']]}")
print(f"  Amounts: {tx_sample['amounts_out']}")

### 2.2 Building a Directed Transaction Graph

In [ ]:
G = nx.MultiDiGraph()  # MultiDiGraph allows parallel edges

for tx in synthetic_txs:
    for inp in tx['inputs']:
        for j, out in enumerate(tx['outputs']):
            G.add_edge(inp, out, txid=tx['txid'], amount=tx['amounts_out'][j])

print(f"Transaction Graph Statistics:")
print(f"  Nodes (addresses): {G.number_of_nodes()}")
print(f"  Edges (flows):     {G.number_of_edges()}")
print(f"  Density:           {nx.density(G):.4f}")

### 2.3 Visualizing the Transaction Graph

In [ ]:
# Assign colors by user
user_colors = {'Alice': '#e74c3c', 'Bob': '#3498db', 'Carol': '#2ecc71',
               'Dave': '#f39c12', 'Eve': '#9b59b6'}

node_colors = [user_colors[address_owner[n]] for n in G.nodes()]

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
pos = nx.spring_layout(G, seed=42, k=2)

# Draw edges
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.3, arrows=True,
                       arrowsize=15, edge_color='gray',
                       connectionstyle='arc3,rad=0.1')

# Draw nodes
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                       node_size=400, alpha=0.9, edgecolors='black', linewidths=0.5)

# Labels: short address
labels = {n: n[1:6] for n in G.nodes()}
nx.draw_networkx_labels(G, pos, labels, font_size=7, ax=ax)

# Legend
patches = [mpatches.Patch(color=c, label=u) for u, c in user_colors.items()]
ax.legend(handles=patches, loc='upper left', title='User (Ground Truth)')
ax.set_title('Transaction Graph -- Colored by True Owner', fontsize=14)
plt.tight_layout()
plt.show()

### 2.4 Degree Centrality and Hub Detection

In [ ]:
# Compute degree centrality on the simplified DiGraph
G_simple = nx.DiGraph(G)  # collapse parallel edges
in_deg = dict(G_simple.in_degree())
out_deg = dict(G_simple.out_degree())
centrality = nx.degree_centrality(G_simple)

# Build a summary DataFrame
rows = []
for node in G_simple.nodes():
    rows.append({
        'address': node[:20] + '...',
        'owner': address_owner[node],
        'in_degree': in_deg[node],
        'out_degree': out_deg[node],
        'total_degree': in_deg[node] + out_deg[node],
        'centrality': round(centrality[node], 4),
    })

df_centrality = pd.DataFrame(rows).sort_values('centrality', ascending=False)

print("Address Centrality Analysis (top 10):")
print("="*75)
print(df_centrality.head(10).to_string(index=False))

# Identify hub nodes (centrality > mean + 1 std)
threshold = df_centrality['centrality'].mean() + df_centrality['centrality'].std()
hubs = df_centrality[df_centrality['centrality'] >= threshold]
print(f"\nHub threshold (mean + 1 std): {threshold:.4f}")
print(f"Hub addresses: {len(hubs)}")
for _, row in hubs.iterrows():
    print(f"  {row['address']}  owner={row['owner']}  centrality={row['centrality']}")

---
# Part 3: Address Clustering

Address clustering groups addresses that are likely controlled by the same entity. The most common heuristic is **multi-input clustering** based on CIOH, implemented efficiently with a **Union-Find** (disjoint set) data structure.

### 3.1 Union-Find Data Structure

In [ ]:
class UnionFind:
    """Disjoint Set Union with path compression and union by rank."""
    
    def __init__(self):
        self.parent = {}
        self.rank = {}
    
    def find(self, x):
        """Find root of x with path compression."""
        if x not in self.parent:
            self.parent[x] = x
            self.rank[x] = 0
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # path compression
        return self.parent[x]
    
    def union(self, x, y):
        """Merge the sets containing x and y."""
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return
        # Union by rank
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
    
    def get_clusters(self) -> Dict[str, Set[str]]:
        """Return a dict mapping root -> set of members."""
        clusters = defaultdict(set)
        for x in self.parent:
            clusters[self.find(x)].add(x)
        return dict(clusters)

# Quick test
uf_test = UnionFind()
uf_test.union('A', 'B')
uf_test.union('B', 'C')
uf_test.union('D', 'E')
print("Union-Find test clusters:", uf_test.get_clusters())
print("find('C') == find('A')?", uf_test.find('C') == uf_test.find('A'))

### 3.2 Multi-Input Clustering with CIOH

In [ ]:
def multi_input_clustering(transactions: list) -> UnionFind:
    """Cluster addresses using CIOH on multi-input transactions."""
    uf = UnionFind()
    
    # Ensure all addresses are registered
    for tx in transactions:
        for addr in tx['inputs'] + tx['outputs']:
            uf.find(addr)
    
    # Apply CIOH: union all inputs in the same transaction
    multi_input_count = 0
    for tx in transactions:
        if len(tx['inputs']) > 1:
            multi_input_count += 1
            first = tx['inputs'][0]
            for inp in tx['inputs'][1:]:
                uf.union(first, inp)
    
    print(f"Processed {len(transactions)} transactions.")
    print(f"Multi-input transactions used for clustering: {multi_input_count}")
    return uf

uf = multi_input_clustering(synthetic_txs)
clusters = uf.get_clusters()

print(f"\nTotal clusters found: {len(clusters)}")
print("\nCluster details:")
for i, (root, members) in enumerate(sorted(clusters.items(), key=lambda x: -len(x[1]))):
    owners = set(address_owner[m] for m in members)
    print(f"  Cluster {i+1}: {len(members)} address(es), true owner(s): {owners}")
    for m in members:
        print(f"    {m[:24]}...  ({address_owner[m]})")

### 3.3 Cluster Visualization

In [ ]:
# Assign cluster IDs for coloring
cluster_id = {}
for cid, (root, members) in enumerate(clusters.items()):
    for m in members:
        cluster_id[m] = cid

# Color by inferred cluster
cmap = plt.cm.get_cmap('tab10', len(clusters))
inferred_colors = [cmap(cluster_id[n]) for n in G.nodes()]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: true owners
nx.draw_networkx_edges(G, pos, ax=axes[0], alpha=0.2, arrows=True, arrowsize=10,
                       connectionstyle='arc3,rad=0.1')
nx.draw_networkx_nodes(G, pos, ax=axes[0], node_color=node_colors,
                       node_size=350, edgecolors='black', linewidths=0.5)
nx.draw_networkx_labels(G, pos, labels, font_size=6, ax=axes[0])
axes[0].set_title('Ground Truth (by Owner)', fontsize=13)

# Right: inferred clusters
nx.draw_networkx_edges(G, pos, ax=axes[1], alpha=0.2, arrows=True, arrowsize=10,
                       connectionstyle='arc3,rad=0.1')
nx.draw_networkx_nodes(G, pos, ax=axes[1], node_color=inferred_colors,
                       node_size=350, edgecolors='black', linewidths=0.5)
nx.draw_networkx_labels(G, pos, labels, font_size=6, ax=axes[1])
axes[1].set_title('Inferred Clusters (CIOH)', fontsize=13)

plt.suptitle('Address Clustering: Ground Truth vs. CIOH Inference', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 3.4 Cluster Size Distribution

In [ ]:
cluster_sizes = [len(members) for members in clusters.values()]
size_counts = Counter(cluster_sizes)

print("Cluster Size Distribution:")
for size in sorted(size_counts):
    print(f"  Size {size}: {size_counts[size]} cluster(s)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(size_counts.keys(), size_counts.values(), color='steelblue', edgecolor='black')
ax.set_xlabel('Cluster Size (number of addresses)')
ax.set_ylabel('Number of Clusters')
ax.set_title('Address Cluster Size Distribution')
ax.set_xticks(sorted(size_counts.keys()))
plt.tight_layout()
plt.show()

---
# Part 4: Taint Analysis

Taint analysis traces the flow of "tainted" funds (e.g., from a known theft) through the transaction graph. Different methods yield different results, which has legal and ethical implications.

We implement two methods:
1. **Poison / Haircut method** -- taint is distributed proportionally across outputs.
2. **FIFO method** -- taint follows a first-in-first-out ordering of inputs.

### 4.1 A 5-Transaction Taint Chain

We create a focused chain of 5 transactions starting from a tainted source.

In [ ]:
# Simple addresses for clarity
taint_chain = [
    {  # TX 0: Tainted source sends 10 BTC
        'txid': 'taint_tx0',
        'inputs': [('TAINTED_SOURCE', 10.0)],
        'outputs': [('Addr_A', 7.0), ('Addr_B', 3.0)],
    },
    {  # TX 1: Addr_A combines with clean funds
        'txid': 'taint_tx1',
        'inputs': [('Addr_A', 7.0), ('CLEAN_1', 5.0)],
        'outputs': [('Addr_C', 8.0), ('Addr_D', 4.0)],
    },
    {  # TX 2: Addr_C spends
        'txid': 'taint_tx2',
        'inputs': [('Addr_C', 8.0)],
        'outputs': [('Addr_E', 5.0), ('Addr_F', 3.0)],
    },
    {  # TX 3: Addr_B combines with clean funds
        'txid': 'taint_tx3',
        'inputs': [('Addr_B', 3.0), ('CLEAN_2', 7.0)],
        'outputs': [('Addr_G', 6.0), ('Addr_H', 4.0)],
    },
    {  # TX 4: Addr_E and Addr_G combine
        'txid': 'taint_tx4',
        'inputs': [('Addr_E', 5.0), ('Addr_G', 6.0)],
        'outputs': [('Addr_I', 7.0), ('Addr_J', 4.0)],
    },
]

print("Taint Chain Transactions:")
print("="*65)
for tx in taint_chain:
    ins = ', '.join(f"{a}({v})" for a, v in tx['inputs'])
    outs = ', '.join(f"{a}({v})" for a, v in tx['outputs'])
    print(f"  {tx['txid']}: [{ins}] -> [{outs}]")

### 4.2 Poison / Haircut Method

In the haircut method, taint from each input is distributed to outputs **proportionally** to their value.

In [ ]:
def haircut_taint_analysis(chain: list, initial_taint: dict) -> dict:
    """
    Haircut/proportional taint analysis.
    initial_taint: {address: taint_fraction} for source addresses.
    Returns: {address: taint_fraction} for all addresses.
    """
    taint = dict(initial_taint)  # taint as fraction (0.0 to 1.0)
    
    for tx in chain:
        # Calculate total tainted value flowing in
        total_in = sum(v for _, v in tx['inputs'])
        tainted_in = sum(v * taint.get(a, 0.0) for a, v in tx['inputs'])
        
        if tainted_in == 0:
            # No taint flows through this transaction
            for a, v in tx['outputs']:
                taint.setdefault(a, 0.0)
            continue
        
        # Taint fraction of total input
        taint_ratio = tainted_in / total_in
        
        # Distribute proportionally to all outputs
        for addr, value in tx['outputs']:
            taint[addr] = taint_ratio  # each output gets the same taint ratio
    
    return taint

# TAINTED_SOURCE is 100% tainted, clean sources are 0%
initial_taint = {'TAINTED_SOURCE': 1.0, 'CLEAN_1': 0.0, 'CLEAN_2': 0.0}

haircut_result = haircut_taint_analysis(taint_chain, initial_taint)

print("Haircut Method -- Taint Fractions:")
print("="*45)
for addr in sorted(haircut_result, key=lambda x: -haircut_result[x]):
    pct = haircut_result[addr] * 100
    bar = '#' * int(pct // 2)
    print(f"  {addr:18s}  {pct:6.1f}%  {bar}")

### 4.3 FIFO Method

In the FIFO method, tainted inputs are consumed first, and taint flows to outputs in order.

In [ ]:
def fifo_taint_analysis(chain: list, initial_taint: dict) -> dict:
    """
    FIFO taint analysis: tainted inputs fill outputs in order.
    Returns: {address: taint_fraction} for all addresses.
    """
    taint = dict(initial_taint)
    
    for tx in chain:
        # Sort inputs: tainted first (higher taint fraction first)
        inputs_sorted = sorted(tx['inputs'],
                               key=lambda x: -taint.get(x[0], 0.0))
        
        # Build a queue of (amount, taint_flag) chunks
        chunks = []
        for addr, value in inputs_sorted:
            t = taint.get(addr, 0.0)
            tainted_amount = value * t
            clean_amount = value * (1 - t)
            if tainted_amount > 0:
                chunks.append((tainted_amount, True))
            if clean_amount > 0:
                chunks.append((clean_amount, False))
        
        # Fill outputs in order from the chunk queue
        chunk_idx = 0
        chunk_remaining = chunks[0][0] if chunks else 0
        chunk_tainted = chunks[0][1] if chunks else False
        
        for addr, value in tx['outputs']:
            tainted_value = 0.0
            remaining = value
            
            while remaining > 1e-10 and chunk_idx < len(chunks):
                take = min(remaining, chunk_remaining)
                if chunk_tainted:
                    tainted_value += take
                remaining -= take
                chunk_remaining -= take
                
                if chunk_remaining < 1e-10:
                    chunk_idx += 1
                    if chunk_idx < len(chunks):
                        chunk_remaining = chunks[chunk_idx][0]
                        chunk_tainted = chunks[chunk_idx][1]
            
            taint[addr] = tainted_value / value if value > 0 else 0.0
    
    return taint

fifo_result = fifo_taint_analysis(taint_chain, initial_taint)

print("FIFO Method -- Taint Fractions:")
print("="*45)
for addr in sorted(fifo_result, key=lambda x: -fifo_result[x]):
    pct = fifo_result[addr] * 100
    bar = '#' * int(pct // 2)
    print(f"  {addr:18s}  {pct:6.1f}%  {bar}")

### 4.4 Comparing Taint Methods

In [ ]:
# Compare the two methods side by side
output_addrs = ['Addr_A', 'Addr_B', 'Addr_C', 'Addr_D', 'Addr_E', 'Addr_F',
                'Addr_G', 'Addr_H', 'Addr_I', 'Addr_J']

comparison = pd.DataFrame({
    'Address': output_addrs,
    'Haircut (%)': [round(haircut_result.get(a, 0) * 100, 1) for a in output_addrs],
    'FIFO (%)': [round(fifo_result.get(a, 0) * 100, 1) for a in output_addrs],
})
comparison['Difference (pp)'] = abs(comparison['Haircut (%)'] - comparison['FIFO (%)'])

print("Taint Method Comparison:")
print("="*55)
print(comparison.to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(output_addrs))
width = 0.35

ax.bar(x - width/2, comparison['Haircut (%)'], width, label='Haircut', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, comparison['FIFO (%)'], width, label='FIFO', color='#3498db', alpha=0.8)

ax.set_xlabel('Address')
ax.set_ylabel('Taint (%)')
ax.set_title('Taint Analysis: Haircut vs. FIFO Method')
ax.set_xticks(x)
ax.set_xticklabels(output_addrs, rotation=45)
ax.legend()
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% threshold')
plt.tight_layout()
plt.show()

print("\nKey Insight: The haircut method spreads taint more evenly, while")
print("FIFO concentrates taint in earlier outputs. The choice of method")
print("has significant implications for identifying 'tainted' funds.")

### 4.5 Taint Flow Visualization

In [ ]:
# Build a directed graph for the taint chain
G_taint = nx.DiGraph()

all_taint_addrs = set()
for tx in taint_chain:
    for a, v in tx['inputs']:
        all_taint_addrs.add(a)
    for a, v in tx['outputs']:
        all_taint_addrs.add(a)
        for ia, iv in tx['inputs']:
            G_taint.add_edge(ia, a, txid=tx['txid'])

# Color by haircut taint level
taint_colors = []
for n in G_taint.nodes():
    t = haircut_result.get(n, 0)
    # Red for tainted, green for clean
    taint_colors.append((t, 1 - t, 0.2))

pos_taint = nx.spring_layout(G_taint, seed=123, k=2.5)

fig, ax = plt.subplots(figsize=(12, 7))
nx.draw_networkx_edges(G_taint, pos_taint, ax=ax, alpha=0.5, arrows=True,
                       arrowsize=15, edge_color='gray')
nx.draw_networkx_nodes(G_taint, pos_taint, ax=ax, node_color=taint_colors,
                       node_size=600, edgecolors='black', linewidths=1)
nx.draw_networkx_labels(G_taint, pos_taint, font_size=8, ax=ax)

ax.set_title('Taint Flow Visualization (Haircut Method)\nRed = Tainted, Green = Clean', fontsize=13)
plt.tight_layout()
plt.show()

---
# Part 5: Privacy Techniques

This section explores techniques that enhance transaction privacy: **CoinJoin** for transaction-level mixing and **hash-based commitment schemes** for hiding values until a reveal phase.

### 5.1 CoinJoin Simulation

CoinJoin is a technique where **multiple participants combine their inputs and outputs into a single transaction** with equal-value outputs. This breaks the Common-Input-Ownership Heuristic because the inputs now belong to different users.

In [ ]:
def simulate_coinjoin(participants: list, equal_amount: float) -> dict:
    """
    Simulate a CoinJoin transaction.
    participants: list of dicts with 'name', 'input_addr', 'output_addr', 'input_amount'
    equal_amount: the standardized output amount for each participant
    """
    inputs = []
    outputs = []
    change_outputs = []
    
    for p in participants:
        inputs.append((p['input_addr'], p['input_amount']))
        outputs.append((p['output_addr'], equal_amount))
        
        # Change back to a fresh address
        change = round(p['input_amount'] - equal_amount - 0.0001, 4)  # fee per participant
        if change > 0:
            change_addr = generate_address(f"{p['name']}-change".encode())
            change_outputs.append((change_addr, change))
    
    # Shuffle outputs to hide which output belongs to which input
    all_outputs = outputs + change_outputs
    random.shuffle(all_outputs)
    
    return {
        'txid': 'coinjoin_tx_001',
        'inputs': inputs,
        'outputs': all_outputs,
        'equal_amount': equal_amount,
        'n_participants': len(participants),
    }

# 3 participants with different input amounts, equal output of 0.5 BTC
cj_participants = [
    {'name': 'Alice', 'input_addr': alice_addrs[2],
     'output_addr': generate_address(b'alice-cj-out'), 'input_amount': 0.8},
    {'name': 'Bob', 'input_addr': bob_addrs[2],
     'output_addr': generate_address(b'bob-cj-out'), 'input_amount': 0.65},
    {'name': 'Carol',
     'input_addr': [a for a in addresses if address_owner[a] == 'Carol'][2],
     'output_addr': generate_address(b'carol-cj-out'), 'input_amount': 0.72},
]

cj_tx = simulate_coinjoin(cj_participants, equal_amount=0.5)

print("CoinJoin Transaction:")
print("="*65)
print(f"TXID: {cj_tx['txid']}")
print(f"Participants: {cj_tx['n_participants']}")
print(f"\nInputs:")
for addr, amt in cj_tx['inputs']:
    print(f"  {addr[:24]}...  {amt} BTC")
print(f"\nOutputs (shuffled):")
for addr, amt in cj_tx['outputs']:
    marker = " <-- equal amount" if abs(amt - 0.5) < 0.001 else " (change)"
    print(f"  {addr[:24]}...  {amt} BTC{marker}")

### 5.2 CoinJoin Breaks CIOH

In [ ]:
# Apply CIOH to the CoinJoin transaction
cj_as_tx = {
    'txid': cj_tx['txid'],
    'inputs': [addr for addr, _ in cj_tx['inputs']],
    'outputs': [addr for addr, _ in cj_tx['outputs']],
    'amounts_out': [amt for _, amt in cj_tx['outputs']],
}

cioh_on_cj = apply_cioh([cj_as_tx])

print("CIOH Applied to CoinJoin Transaction:")
print("="*55)
if cioh_on_cj:
    for group in cioh_on_cj:
        print(f"\nCIOH concludes these {len(group)} addresses belong to ONE entity:")
        for a in group:
            true_owner = address_owner.get(a, 'Unknown')
            print(f"  {a[:24]}...  (true owner: {true_owner})")

print("\n*** CIOH FAILURE: The heuristic incorrectly groups addresses")
print("    from 3 different users into one cluster. This is exactly")
print("    why CoinJoin is an effective privacy technique. ***")

### 5.3 Hash-Based Commitment Scheme

A commitment scheme allows a party to **commit** to a value without revealing it, then **reveal** and **verify** later. This is fundamental to many privacy protocols.

$$\text{commit}(v, r) = \text{SHA256}(v \| r)$$

where $v$ is the value and $r$ is a random nonce.

In [ ]:
class CommitmentScheme:
    """Hash-based commitment scheme using SHA-256."""
    
    @staticmethod
    def commit(value: str, nonce: bytes = None) -> Tuple[str, bytes]:
        """Create a commitment to a value.
        Returns: (commitment_hash, nonce)
        """
        if nonce is None:
            nonce = secrets.token_bytes(32)
        message = value.encode() + nonce
        commitment = hashlib.sha256(message).hexdigest()
        return commitment, nonce
    
    @staticmethod
    def verify(value: str, nonce: bytes, commitment: str) -> bool:
        """Verify that a value matches a commitment."""
        message = value.encode() + nonce
        computed = hashlib.sha256(message).hexdigest()
        return computed == commitment

cs = CommitmentScheme()

# Demonstrate the commit-reveal-verify cycle
print("=== Commit-Reveal-Verify Cycle ===")
print()

# Step 1: Commit
secret_value = "I will pay 2.5 BTC"
commitment, nonce = cs.commit(secret_value)
print(f"Step 1 - COMMIT:")
print(f"  Secret value: '{secret_value}'")
print(f"  Nonce:        {nonce.hex()[:32]}...")
print(f"  Commitment:   {commitment}")
print(f"  (Only the commitment is published; value and nonce are kept secret.)")

# Step 2: Reveal
print(f"\nStep 2 - REVEAL:")
print(f"  The committer reveals: value='{secret_value}', nonce={nonce.hex()[:32]}...")

# Step 3: Verify
is_valid = cs.verify(secret_value, nonce, commitment)
print(f"\nStep 3 - VERIFY:")
print(f"  Recompute SHA256(value || nonce) and compare to commitment.")
print(f"  Valid: {is_valid}")

# Test with wrong value
is_valid_wrong = cs.verify("I will pay 0.1 BTC", nonce, commitment)
print(f"\n  Verify with tampered value: {is_valid_wrong}")
print(f"  -> Binding property: committer cannot change their mind.")

### 5.4 Anonymity Set Analysis

The **anonymity set** is the set of possible senders (or receivers) for a given transaction. Larger anonymity sets mean stronger privacy. We calculate anonymity set sizes for different transaction types.

In [ ]:
def calculate_anonymity_set(tx: dict) -> dict:
    """
    Calculate anonymity set metrics for a transaction.
    Returns dict with sender and receiver anonymity set sizes.
    """
    n_inputs = len(tx.get('inputs', []))
    n_outputs = len(tx.get('outputs', []))
    
    # For standard tx: anonymity set = 1 (trivially linkable)
    # For CoinJoin: anonymity set = number of equal-value outputs
    amounts_out = tx.get('amounts_out', [])
    if not amounts_out:
        amounts_out = [amt for _, amt in tx.get('outputs', [])]
    
    # Count equal-value output groups
    amount_counts = Counter(round(a, 4) for a in amounts_out)
    max_equal = max(amount_counts.values()) if amount_counts else 1
    
    # Entropy-based anonymity metric
    total = sum(amount_counts.values())
    entropy = 0
    for count in amount_counts.values():
        p = count / total
        if p > 0:
            entropy -= p * np.log2(p)
    
    return {
        'n_inputs': n_inputs,
        'n_outputs': n_outputs,
        'max_equal_outputs': max_equal,
        'anonymity_set_size': max_equal,
        'entropy_bits': round(entropy, 3),
    }

# Compare: standard transaction vs CoinJoin
standard_tx_example = {
    'inputs': ['sender_addr'],
    'outputs': [('receiver', 1.0), ('change', 0.5)],
    'amounts_out': [1.0, 0.5],
}

cj_amounts = [amt for _, amt in cj_tx['outputs']]
cj_analysis_tx = {
    'inputs': [addr for addr, _ in cj_tx['inputs']],
    'outputs': cj_tx['outputs'],
    'amounts_out': cj_amounts,
}

std_metrics = calculate_anonymity_set(standard_tx_example)
cj_metrics = calculate_anonymity_set(cj_analysis_tx)

print("Anonymity Set Comparison:")
print("="*55)
metrics_df = pd.DataFrame([
    {'Type': 'Standard TX', **std_metrics},
    {'Type': 'CoinJoin (3 participants)', **cj_metrics},
])
print(metrics_df.to_string(index=False))

print(f"\nThe CoinJoin's anonymity set of {cj_metrics['anonymity_set_size']} means an observer")
print(f"cannot determine which of {cj_metrics['anonymity_set_size']} equal outputs belongs to which input.")

---
# Exercises

These exercises build on the concepts covered above. For background reading, see [Section 6: Privacy Technologies](../sections/06-privacy-technologies.md).

## Exercise 1: Change Address Detection Heuristic

In a standard Bitcoin transaction with two outputs, one output is the payment and the other is **change** returned to the sender. Implement a heuristic to detect the change output.

**Heuristic rules:**
1. If one output amount is a round number and the other is not, the non-round amount is likely change.
2. If one output address has appeared before (in inputs) and the other has not, the new address is likely change.
3. If one output is significantly smaller than the other, the smaller one is likely change.

Implement the function below and test it on the provided transactions.

In [ ]:
def detect_change_output(tx: dict, known_addresses: set) -> dict:
    """
    Detect which output is likely the change output.
    
    Args:
        tx: dict with 'inputs' (list of addrs), 'outputs' (list of addrs),
            'amounts_out' (list of floats)
        known_addresses: set of addresses that have appeared before
    
    Returns:
        dict with 'change_index', 'change_address', 'confidence', 'reason'
    """
    if len(tx['outputs']) != 2:
        return {'change_index': None, 'confidence': 0, 'reason': 'Not a 2-output tx'}
    
    scores = [0, 0]  # Score for each output being change
    reasons = [[], []]
    
    # Rule 1: Round number detection
    for i in range(2):
        amt = tx['amounts_out'][i]
        # Check if amount is a round number (e.g., 1.0, 0.5, 0.1)
        if amt * 10 == int(amt * 10):  # round to 1 decimal
            # Round numbers are less likely to be change
            scores[1 - i] += 1
            reasons[1 - i].append('other output is round number')
    
    # Rule 2: Address reuse -- known addresses are less likely change addresses
    for i in range(2):
        if tx['outputs'][i] not in known_addresses and tx['outputs'][1-i] in known_addresses:
            scores[i] += 2  # new address more likely to be change
            reasons[i].append('new address (not seen before)')
    
    # Rule 3: Smaller output is more likely change
    if tx['amounts_out'][0] < tx['amounts_out'][1]:
        scores[0] += 1
        reasons[0].append('smaller amount')
    elif tx['amounts_out'][1] < tx['amounts_out'][0]:
        scores[1] += 1
        reasons[1].append('smaller amount')
    
    change_idx = 0 if scores[0] >= scores[1] else 1
    max_score = max(scores)
    confidence = min(max_score / 4.0, 1.0)  # Normalize to 0-1
    
    return {
        'change_index': change_idx,
        'change_address': tx['outputs'][change_idx],
        'confidence': round(confidence, 2),
        'reason': '; '.join(reasons[change_idx]) if reasons[change_idx] else 'default'
    }

# Test transactions
known_addrs = set(addresses[:6])  # First 6 addresses are "known"

test_txs = [
    {
        'txid': 'test_1',
        'inputs': [addresses[0]],
        'outputs': [addresses[3], generate_address(b'change-test-1')],
        'amounts_out': [1.0, 0.2347],
    },
    {
        'txid': 'test_2',
        'inputs': [addresses[1]],
        'outputs': [generate_address(b'new-addr-1'), addresses[4]],
        'amounts_out': [0.1532, 0.5],
    },
    {
        'txid': 'test_3',
        'inputs': [addresses[2]],
        'outputs': [generate_address(b'new-addr-2'), generate_address(b'new-addr-3')],
        'amounts_out': [2.0, 0.0891],
    },
]

print("Change Address Detection Results:")
print("="*65)
for tx in test_txs:
    result = detect_change_output(tx, known_addrs)
    print(f"\n{tx['txid']}:")
    print(f"  Output 0: {tx['outputs'][0][:20]}...  {tx['amounts_out'][0]} BTC")
    print(f"  Output 1: {tx['outputs'][1][:20]}...  {tx['amounts_out'][1]} BTC")
    print(f"  Detected change: Output {result['change_index']}")
    print(f"  Confidence: {result['confidence']}")
    print(f"  Reason: {result['reason']}")

## Exercise 2: Privacy Score Calculator

Design a function that computes a **privacy score** (0 to 100) for a transaction based on multiple factors:
- Address reuse (negative)
- Number of equal-value outputs (positive)
- Number of inputs from different clusters (positive -- suggests CoinJoin)
- Round output amounts (negative -- makes change detection easier)

Implement the scorer and evaluate several example transactions.

In [ ]:
def privacy_score(tx: dict, address_history: set, cluster_fn=None) -> dict:
    """
    Calculate a privacy score for a transaction (0-100).
    
    Args:
        tx: dict with 'inputs', 'outputs', 'amounts_out'
        address_history: set of previously-seen addresses
        cluster_fn: optional function(addr) -> cluster_id
    
    Returns:
        dict with 'score', 'breakdown', 'recommendations'
    """
    score = 50  # Start at neutral
    breakdown = {}
    recommendations = []
    
    inputs = tx['inputs']
    outputs = tx['outputs']
    amounts = tx['amounts_out']
    
    # Factor 1: Address reuse (-20 if any output reuses an input address)
    input_set = set(inputs)
    reused = input_set.intersection(set(outputs))
    if reused:
        score -= 20
        breakdown['address_reuse'] = -20
        recommendations.append('Avoid sending change to an input address')
    else:
        breakdown['address_reuse'] = 0
    
    # Factor 2: Equal-value outputs (+15 per additional equal output)
    amount_counts = Counter(round(a, 4) for a in amounts)
    max_equal = max(amount_counts.values())
    equal_bonus = min((max_equal - 1) * 15, 30)
    score += equal_bonus
    breakdown['equal_outputs'] = equal_bonus
    if max_equal < 2:
        recommendations.append('Use equal-value outputs (CoinJoin) for better privacy')
    
    # Factor 3: Multiple input clusters (+10 if inputs span 2+ clusters)
    if cluster_fn and len(inputs) > 1:
        cluster_ids = set(cluster_fn(a) for a in inputs)
        if len(cluster_ids) > 1:
            score += 10
            breakdown['multi_cluster_inputs'] = 10
        else:
            breakdown['multi_cluster_inputs'] = 0
    else:
        breakdown['multi_cluster_inputs'] = 0
    
    # Factor 4: Round amounts penalty (-5 per round amount)
    round_count = sum(1 for a in amounts if a * 10 == int(a * 10))
    round_penalty = min(round_count * 5, 15)
    score -= round_penalty
    breakdown['round_amounts'] = -round_penalty
    if round_count > 0:
        recommendations.append('Avoid round output amounts to hinder change detection')
    
    # Factor 5: Known address in outputs (-10)
    known_in_outputs = set(outputs).intersection(address_history)
    if known_in_outputs:
        score -= 10
        breakdown['known_output_addrs'] = -10
        recommendations.append('Use fresh addresses for outputs')
    else:
        breakdown['known_output_addrs'] = 0
    
    score = max(0, min(100, score))
    
    return {
        'score': score,
        'grade': 'A' if score >= 80 else 'B' if score >= 60 else 'C' if score >= 40 else 'D' if score >= 20 else 'F',
        'breakdown': breakdown,
        'recommendations': recommendations,
    }

# Test on different transaction types
history = set(addresses[:10])

# Bad privacy: reuses address, round amounts
bad_tx = {
    'inputs': [addresses[0]],
    'outputs': [addresses[3], addresses[0]],  # change back to input addr!
    'amounts_out': [1.0, 0.5],
}

# Good privacy: CoinJoin-like, fresh addresses
good_tx = {
    'inputs': [addresses[0], addresses[6]],  # different owners
    'outputs': [generate_address(b'fresh-1'), generate_address(b'fresh-2'),
                generate_address(b'fresh-3')],
    'amounts_out': [0.5, 0.5, 0.1234],
}

def simple_cluster(addr):
    return address_owner.get(addr, addr)

for label, tx in [('Low Privacy TX', bad_tx), ('High Privacy TX', good_tx)]:
    result = privacy_score(tx, history, cluster_fn=simple_cluster)
    print(f"\n{'='*50}")
    print(f"{label}")
    print(f"  Score: {result['score']}/100  Grade: {result['grade']}")
    print(f"  Breakdown: {result['breakdown']}")
    if result['recommendations']:
        print(f"  Recommendations:")
        for r in result['recommendations']:
            print(f"    - {r}")

## Exercise 3: 5-Participant CoinJoin Anonymity Analysis

Extend the CoinJoin simulation to **5 participants**. Analyze how anonymity set size scales and compute the probability that an observer can correctly link an input to its output.

In [ ]:
# 5-participant CoinJoin
cj5_participants = []
for i, name in enumerate(['Alice', 'Bob', 'Carol', 'Dave', 'Eve']):
    cj5_participants.append({
        'name': name,
        'input_addr': generate_address(f'{name}-cj5-in'.encode()),
        'output_addr': generate_address(f'{name}-cj5-out'.encode()),
        'input_amount': round(random.uniform(0.6, 1.5), 4),
    })

cj5_tx = simulate_coinjoin(cj5_participants, equal_amount=0.5)

print("5-Participant CoinJoin:")
print("="*65)
print(f"Inputs:  {len(cj5_tx['inputs'])}")
print(f"Outputs: {len(cj5_tx['outputs'])}")

# Count equal-value outputs
equal_outputs = sum(1 for _, amt in cj5_tx['outputs'] if abs(amt - 0.5) < 0.001)
change_outputs = len(cj5_tx['outputs']) - equal_outputs

print(f"\nEqual-value outputs (0.5 BTC): {equal_outputs}")
print(f"Change outputs (various):       {change_outputs}")
print(f"\nAnonymity set size: {equal_outputs}")
print(f"Probability of correct link (random guess): 1/{equal_outputs} = {1/equal_outputs:.1%}")

# Show how anonymity scales with participants
print("\n--- Anonymity Set Scaling ---")
participant_counts = [2, 3, 5, 10, 20, 50, 100]
for n in participant_counts:
    prob = 1 / n
    entropy = np.log2(n)
    print(f"  {n:3d} participants -> anonymity set = {n:3d}, "
          f"guess prob = {prob:.4f}, entropy = {entropy:.2f} bits")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(participant_counts, [1/n for n in participant_counts], 'ro-')
ax1.set_xlabel('Number of CoinJoin Participants')
ax1.set_ylabel('Probability of Correct Link')
ax1.set_title('Deanonymization Probability')
ax1.set_xscale('log')
ax1.grid(True, alpha=0.3)

ax2.plot(participant_counts, [np.log2(n) for n in participant_counts], 'bs-')
ax2.set_xlabel('Number of CoinJoin Participants')
ax2.set_ylabel('Anonymity Entropy (bits)')
ax2.set_title('Anonymity Entropy Scaling')
ax2.set_xscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercise 4: Ring Signature Concept Demonstration (Simplified)

A **ring signature** allows a signer to produce a signature on behalf of a group ("ring") without revealing which member actually signed. This is used in Monero for sender privacy.

Below is a simplified conceptual demonstration using hash-based commitments to illustrate the key properties:
1. **Unforgeability** -- only a ring member can produce a valid signature.
2. **Anonymity** -- a verifier cannot determine which member signed.
3. **Linkability** (optional) -- using a key image to prevent double-spending.

In [ ]:
class SimplifiedRingSignature:
    """
    Simplified ring signature demonstration.
    
    This is NOT a cryptographically secure implementation. It demonstrates
    the conceptual properties of ring signatures using hash-based simulation.
    Real ring signatures use elliptic curve algebra.
    """
    
    def __init__(self, ring_size: int):
        """Generate a ring of key pairs."""
        self.ring_size = ring_size
        self.private_keys = [secrets.token_bytes(32) for _ in range(ring_size)]
        self.public_keys = [
            hashlib.sha256(sk).hexdigest() for sk in self.private_keys
        ]
    
    def sign(self, message: str, signer_index: int) -> dict:
        """
        Create a ring signature. The signer uses their private key;
        for other members, random values simulate their contribution.
        """
        sk = self.private_keys[signer_index]
        
        # Generate random values for non-signer ring members
        ring_values = []
        for i in range(self.ring_size):
            if i == signer_index:
                # Actual signer: compute from private key + message
                v = hashlib.sha256(sk + message.encode()).hexdigest()
            else:
                # Non-signer: random value
                v = secrets.token_hex(32)
            ring_values.append(v)
        
        # Compute the ring: chain of hashes
        challenge = hashlib.sha256(message.encode()).hexdigest()
        ring_hash = challenge
        for v in ring_values:
            ring_hash = hashlib.sha256(
                (ring_hash + v).encode()
            ).hexdigest()
        
        # Key image (for linkability / double-spend detection)
        key_image = hashlib.sha256(
            sk + b'key_image'
        ).hexdigest()
        
        return {
            'message': message,
            'ring_values': ring_values,
            'ring_hash': ring_hash,
            'key_image': key_image,
            'public_keys': self.public_keys,
        }
    
    def verify(self, signature: dict) -> bool:
        """
        Verify a ring signature by recomputing the ring hash.
        A verifier can confirm validity but NOT identify the signer.
        """
        challenge = hashlib.sha256(signature['message'].encode()).hexdigest()
        ring_hash = challenge
        for v in signature['ring_values']:
            ring_hash = hashlib.sha256(
                (ring_hash + v).encode()
            ).hexdigest()
        return ring_hash == signature['ring_hash']

# Demonstrate
ring = SimplifiedRingSignature(ring_size=5)

print("Ring Signature Demonstration (5 members):")
print("="*55)
print(f"\nPublic keys in the ring:")
for i, pk in enumerate(ring.public_keys):
    print(f"  Member {i}: {pk[:32]}...")

# Member 2 signs (secret)
actual_signer = 2
message = "Transfer 1.5 BTC to Addr_X"
sig = ring.sign(message, actual_signer)

print(f"\nMessage: '{message}'")
print(f"Actual signer: Member {actual_signer} (SECRET - not in signature)")
print(f"Key image: {sig['key_image'][:32]}...")
print(f"Ring hash: {sig['ring_hash'][:32]}...")

# Verify
is_valid = ring.verify(sig)
print(f"\nSignature valid: {is_valid}")
print(f"Verifier knows signer is one of {ring.ring_size} members,")
print(f"but CANNOT determine which one.")

# Demonstrate linkability via key image
print(f"\n--- Linkability / Double-Spend Detection ---")
sig2 = ring.sign("Transfer 2.0 BTC to Addr_Y", actual_signer)
print(f"Signature 1 key image: {sig['key_image'][:32]}...")
print(f"Signature 2 key image: {sig2['key_image'][:32]}...")
print(f"Same key image? {sig['key_image'] == sig2['key_image']}")
print(f"-> Same signer detected (double-spend prevention) without revealing identity.")

# Different signer produces different key image
sig3 = ring.sign("Transfer 0.5 BTC to Addr_Z", 4)
print(f"\nSignature 3 (Member 4) key image: {sig3['key_image'][:32]}...")
print(f"Same as Sig 1? {sig['key_image'] == sig3['key_image']}")
print(f"-> Different signer confirmed.")

---
# Summary

In this notebook, we explored the key concepts of blockchain privacy and forensics:

**Part 1 -- Bitcoin Privacy Model:**
- Bitcoin is pseudonymous, not anonymous. Addresses are unlinkable to identities only if users practice good hygiene.
- Address reuse trivially links transactions. The Common-Input-Ownership Heuristic (CIOH) further links addresses used as inputs in the same transaction.

**Part 2 -- Transaction Graph Analysis:**
- Blockchain transactions form a directed graph that can be analyzed with standard graph algorithms.
- Degree centrality identifies hub addresses (e.g., exchanges). Graph visualization reveals community structure.

**Part 3 -- Address Clustering:**
- The Union-Find data structure enables efficient clustering of addresses via CIOH.
- Multi-input clustering groups addresses that co-appear as inputs, reducing the effective anonymity of the system.

**Part 4 -- Taint Analysis:**
- The haircut/proportional method distributes taint evenly across outputs.
- The FIFO method concentrates taint in earlier outputs.
- The choice of method has significant legal and ethical implications for identifying "tainted" funds.

**Part 5 -- Privacy Techniques:**
- CoinJoin breaks CIOH by combining inputs from multiple users with equal-value outputs.
- Hash-based commitment schemes provide hiding and binding properties.
- Anonymity set size is the key metric for privacy -- larger sets mean stronger privacy.
- Ring signatures (as in Monero) allow signing on behalf of a group without revealing the signer.

**For further reading**, see [Section 6: Privacy Technologies](../sections/06-privacy-technologies.md), which covers zero-knowledge proofs, confidential transactions, and the privacy-scalability tradeoff in greater depth.

---

*Notebook 09 -- Privacy & Forensics in Blockchain*  
*MIT Sloan Blockchain Education Project*